### 랜덤 벽 GridWord에서 TD Learning 구현
랜덤하게 생성되는 벽이 존재하는 GridWord 환경을 만들고, 에이전트가 랜덤 정책으로 움직일 때 TD Learning을 이용해 상태가치 V(s)를 학습하는 프로그램을 작성

- 4*4 GridWorld 환경 구현
- 시작 위치 (0, 0)
- 목표 위치 (3, 3)
- 행동 -> 0:왼쪽, 1:위, 2:오른쪽, 3:아래
- 매 에피소드가 시작될 때마다 격자 안에 벽을 랜덤하게 생성(이동 불가)
    - 벽의 개수는 2개 또는 3개
    - 시작 위치에 벽이 생성되면 안 됨
    - 목표 위치에 벽이 생성되면 안 됨
    - 같은 위치에 벽이 중복 생성되면 안 됨
- 이동하려는 위치가 빈 칸이면 이동 가능
    - 격자를 벗어나면 원래 위치에 그대로 둠
    - 벽이 있는 칸으로 이동하면 제자리 유지
- 보상: 일반 이동 -1
- 목표: (3, 3)에 도착하면 에피소드 종료

> 일반 그리드 환경과, 벽 그리드 환경과의 상태가치 차이에 대해 작성

In [1]:
import random

# =========================
# 1. 환경 정의
# =========================
class GridWorld:
    def __init__(self, size=4, start=(0, 0), goal=(3, 3), step_reward=-1, use_random_walls=False):
        self.size = size
        self.start = start
        self.goal = goal
        self.step_reward = step_reward
        self.use_random_walls = use_random_walls

        self.state = start
        self.walls = set()

        # 행동: 0=왼쪽, 1=위, 2=오른쪽, 3=아래
        self.action_map = {
            0: (0, -1),   # left
            1: (-1, 0),   # up
            2: (0, 1),    # right
            3: (1, 0)     # down
        }

    def all_states(self):
        return [(r, c) for r in range(self.size) for c in range(self.size)]

    def reset(self):
        self.state = self.start

        if self.use_random_walls:
            self._generate_random_walls()
        else:
            self.walls = set()

        return self.state

    def _generate_random_walls(self):
        candidates = [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if (r, c) != self.start and (r, c) != self.goal
        ]

        wall_count = random.choice([2, 3])
        self.walls = set(random.sample(candidates, wall_count))

    def step(self, action):
        cur_r, cur_c = self.state
        dr, dc = self.action_map[action]

        next_r = cur_r + dr
        next_c = cur_c + dc

        # 격자 밖이면 제자리 유지
        if not (0 <= next_r < self.size and 0 <= next_c < self.size):
            next_r, next_c = cur_r, cur_c

        # 벽이면 제자리 유지
        if (next_r, next_c) in self.walls:
            next_r, next_c = cur_r, cur_c

        self.state = (next_r, next_c)

        # 일반 이동 보상 -1
        reward = self.step_reward

        # 목표 도착 시 종료
        done = (self.state == self.goal)

        return self.state, reward, done


# =========================
# 2. TD(0) Learning
# =========================
def td_learning(env, episodes=10000, alpha=0.1, gamma=0.9, max_steps=100):
    """
    랜덤 정책으로 행동하면서 TD(0) 방식으로 V(s) 학습
    V(s) <- V(s) + alpha * [r + gamma * V(s') - V(s)]

    max_steps는 무한 루프 방지용 안전장치
    (랜덤 정책 + 랜덤 벽 환경에서는 목표 도달이 오래 걸리거나 불가능한 맵이 나올 수 있음)
    """
    V = {state: 0.0 for state in env.all_states()}
    V[env.goal] = 0.0  # 종단 상태 가치는 0으로 둠

    for episode in range(episodes):
        state = env.reset()

        for step in range(max_steps):
            if state == env.goal:
                break

            # 랜덤 정책
            action = random.choice([0, 1, 2, 3])

            next_state, reward, done = env.step(action)

            # TD(0) target
            td_target = reward + gamma * V[next_state]
            td_error = td_target - V[state]

            # 업데이트
            V[state] += alpha * td_error

            state = next_state

            if done:
                break

    V[env.goal] = 0.0
    return V


# =========================
# 3. 출력 함수
# =========================
def print_grid_info(env):
    print("현재 벽 배치")
    for r in range(env.size):
        row = []
        for c in range(env.size):
            pos = (r, c)
            if pos == env.start:
                row.append(" S ")
            elif pos == env.goal:
                row.append(" G ")
            elif pos in env.walls:
                row.append(" # ")
            else:
                row.append(" . ")
        print("".join(row))
    print()


def print_value_table(V, env, title="State Value V(s)"):
    print(title)
    for r in range(env.size):
        row = []
        for c in range(env.size):
            pos = (r, c)
            if pos == env.goal:
                row.append("  G:0.00 ")
            else:
                row.append(f"{V[pos]:8.2f}")
        print(" ".join(row))
    print()


def print_difference_table(V_normal, V_wall, env):
    print("차이값 (벽 환경 - 일반 환경)")
    print("값이 더 음수일수록 벽 환경에서 더 불리하다는 뜻")
    for r in range(env.size):
        row = []
        for c in range(env.size):
            pos = (r, c)
            diff = V_wall[pos] - V_normal[pos]
            row.append(f"{diff:8.2f}")
        print(" ".join(row))
    print()


# =========================
# 4. 실행
# =========================
if __name__ == "__main__":
    random.seed(42)

    # 일반 GridWorld
    normal_env = GridWorld(use_random_walls=False)
    V_normal = td_learning(
        env=normal_env,
        episodes=10000,
        alpha=0.1,
        gamma=0.9,
        max_steps=100
    )

    # 랜덤 벽 GridWorld
    wall_env = GridWorld(use_random_walls=True)
    V_wall = td_learning(
        env=wall_env,
        episodes=20000,   # 벽 랜덤성이 있으므로 조금 더 많이 학습
        alpha=0.1,
        gamma=0.9,
        max_steps=100
    )

    # 예시로 마지막 벽 배치 한 번 출력
    wall_env.reset()
    print_grid_info(wall_env)

    # 결과 출력
    print_value_table(V_normal, normal_env, title="일반 GridWorld의 상태가치 V(s)")
    print_value_table(V_wall, wall_env, title="랜덤 벽 GridWorld의 상태가치 V(s)")
    print_difference_table(V_normal, V_wall, normal_env)

현재 벽 배치
 S  .  .  . 
 #  .  .  . 
 .  .  .  # 
 .  .  .  G 

일반 GridWorld의 상태가치 V(s)
   -9.30    -9.20    -8.95    -8.44
   -9.10    -8.88    -8.39    -7.67
   -8.71    -8.29    -7.27    -5.42
   -8.60    -7.48    -5.53   G:0.00 

랜덤 벽 GridWorld의 상태가치 V(s)
   -9.63    -9.61    -9.32    -9.15
   -9.50    -9.26    -9.06    -8.44
   -9.22    -8.55    -7.69    -4.35
   -8.55    -7.84    -5.30   G:0.00 

차이값 (벽 환경 - 일반 환경)
값이 더 음수일수록 벽 환경에서 더 불리하다는 뜻
   -0.33    -0.41    -0.37    -0.71
   -0.40    -0.38    -0.67    -0.76
   -0.51    -0.26    -0.42     1.07
    0.05    -0.36     0.23     0.00

